In [ ]:
# %pip install --upgrade pyhyperscattering
%pip install pyhyperscattering==0.1.6
%pip install hvplot

In [ ]:
# Imports
import PyHyperScattering, pathlib, pickle, hvplot
import numpy as np
import pandas as pd
import xarray as xr
from scipy.ndimage.morphology import binary_erosion
from tiled.client import from_profile
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LogNorm
from matplotlib_inline.backend_inline import set_matplotlib_formats

c = from_profile('rsoxs')
print(f'Using PyHyperScattering Version: {PyHyperScattering.__version__}')

In [ ]:
# Function from Eliot to find scans
def find_scans(inst="",samp="",plan="",num=10):
    search = {'institution':{ '$regex':inst},'sample_name':{ '$regex':samp},'plan_name': { '$regex': plan } }
    results = c.search(search)
    length = len(results)
    print(f"found {length} scans total, showing the last {min(num,max(0,length))}")
    for index in np.arange(-min(num,max(0,length)),0,1):
        index = int(index)
        startdoc = results[index].start
        stopdoc = results[index].stop
        print(f'{startdoc["scan_id"]} - '
              f'{startdoc["institution"]} - '
              f'{startdoc["proposal_id"]} - '
              f'{startdoc["project_name"]} - '
              f'{startdoc["sample_name"]} - '
              f'{startdoc["plan_name"]} - ', end='')
        try:
            print(f'{stopdoc["num_events"]["primary"]}')
        except:
            print('Error')
    
    
# # Function for stacking arrays
# def mask_stacker(mask, Z):
#     """
#     Stacks original (X,Y) array to (X,Y,Z)
#     For Z > 2 (otherwise use np.dstack)
#     """
#     assert Z > 2 and type(Z)==int, 'Z must be integer > 2'
    
#     mask = np.dstack((mask, mask))
    
#     for num in range(Z-2):
#         mask = np.dstack((mask, mask[:,:,0]))
        
#     return mask

In [ ]:
# Define colormap based on terrain colormap (this seems to show RSoXS data well)
# and define what to set values above/below colorbar range (and nan's)

cmap = (plt.cm.terrain).copy()
cmap.set_under('black') # color below the minimum value
cmap.set_over('purple') # color above the maximum value
cmap.set_bad('black') # color for negative numbers (nans on log scale)

cm = plt.cm.terrain.copy()
cm.set_bad('purple')

# Load in pyhyper sst1 file loader
rsoxsload = PyHyperScattering.load.SST1RSoXSDB(corr_mode='none')

In [ ]:
# Set savepaths:
notebookPath = pathlib.Path.cwd()
maskPath = notebookPath.joinpath('masks')
savePath = notebookPath.joinpath('exports')
arrPath = notebookPath.joinpath('pickled_xarrays')
# imagesPath = savePath.joinpath('imgs')
# CvsQimgsPath = imagesPath.joinpath('CvsQ')
# EvsQimgsPath = imagesPath.joinpath('EvsQ')
# rawimgsPath = imagesPath.joinpath('raw')
# ARimgsPath = imagesPath.joinpath('AR')
# lineplotsPath = savePath.joinpath('linecuts')

In [ ]:
find_scans(inst='UColorado', plan='full_carbon_scan_nd', num=30)

In [ ]:
# Define a sample guide if relevant (connect sample ID to detailed name)

# sample_guide = {
#     'andrew1':'PM6-Y6-CF',
#     'andrew2':'PM6-Y6-CFCN',
#     'andrew3':'PM6-Y6-Tol',
#     'andrew4':'PM6-Y7-CF',
#     'andrew5':'PM6-Y7-CFCN',
#     'andrew6':'PM6-Y7-Tol',
#     'andrew7':'PM6-Y7BO-CF',
#     'andrew8':'PM6-Y7BO-CFCN',
#     'andrew9':'PM6-Y7BO-Tol',
#     'andrew10':'PM6-Y12-CF',
#     'andrew11':'PM6-Y12-CFCN',
#     'andrew12':'PM6-Y12-Tol',
#     'andrew13':'PM7D5-Y6-CF',
#     'andrew14':'PM7D5-Y6-Tol',
#     'andrew15':'PM7D5-Y247-CF',
#     'andrew16':'PM7D5-Y247-Tol',
#     'andrew17':'PM7D5-Y12-CF',
#     'andrew18':'PM7D5-Y12-CF',
#     'andrew19':'PM7D5-Y12-Tol',
#     'andrew20':'PM7D5-Y12-Tol'
# }

# sample_guide = {
#     'Blend1':'PM7-Y6-CF',
#     'Blend2':'PM7-Y6-CFCBCN',
#     'Blend3':'PM7-Y247-CF',
#     'Blend4':'PM7-Y247-CFCBCN',
#     'Blend5':'PM7D4-Y6-CF',
#     'Blend6':'PM7D4-Y6-CFCBCN',
#     'Blend7':'PM7D5-Y6-CF',
#     'Blend8':'PM7D5-Y6-CFCBCN',
#     'Blend9':'PM7D3-Y6-CF',
#     'Blend10':'PM7D3-Y246-CF',
#     'Blend11':'PM7D3-Y247-CF',
#     'Blend12':'PM7D3-Y248-CF',
#     'Blend13':'PM7D5-Y246-CF',
#     'Blend14':'PM7D5-Y247-CF',
#     'Blend16':'PM7D5-Y12-OXY',
#     'Blend17':'PM7D5-Y12-2MeTHF',
#     'Blend18':'PM7D5-Y12-CB',
# }

# lengths = []
# for string in list(sample_guide.values()):
#     lengths.append(len(string))

# max_len = max(lengths)
# print(f'Use {max_len} spaces for blend_name while naming folders')

# More just for smaller name for detector for labelling
detector_guide = {
    'Small Angle CCD Detector': 'SAXS',
    'Wide Angle CCD Detector': 'WAXS'
}

In [ ]:
scan_id = 44632

# A is from database, B is from pickled file (pickled file is faster)
loader = 'A'

if loader == 'A':
    raw = rsoxsload.loadRun(run=c[scan_id], dims=['energy'])
    
    sample_name = raw.attrs['sample_name']
    # blend_name = sample_guide[sample_name]
    blend_name = sample_name
    pol = raw.attrs['polarization'][0]
    detector = detector_guide[raw.attrs['detector']]
    
    pickle.dump(raw, arrPath.joinpath(f'raw_{scan_id}_{blend_name:-<9}_{detector}_{int(pol)}deg.pkl').open('wb'), protocol=-1)
elif loader == 'B':
    raw = pickle.load(sorted(arrPath.glob(f'raw_{scan_id}*.pkl'))[0].open('rb'))

In [ ]:
# # Set variables from attributes
# sample_name = raw.attrs['sample_name']
# blend_name = sample_guide[sample_name]
# pol = raw.attrs['polarization'][0]
# detector = detector_guide[raw.attrs['detector']]

# Get beamcenter (rounded to nearest 10) for plotting purposes
bcx_approx = round(raw.attrs['beamcenter_x'], -1)
bcy_approx = round(raw.attrs['beamcenter_y'], -1)

# Create folder for scan to save all processed data:
scanPath = savePath.joinpath(f'{scan_id}_{blend_name:-<9}_{detector}_{int(pol):0>2}deg')
scanPath.mkdir(parents=True, exist_ok=True)

# Show scan at 285eV
energy = 285
title_string = f'{blend_name}, {detector}, {energy} eV, pol = {int(pol)}$^\circ$'
mask_img = raw.unstack('system').sel(energy=energy, method='nearest')
plt.imshow(mask_img.data, origin='lower', norm=LogNorm(1e1, 5e3), cmap=cm)
plt.colorbar()
plt.xlabel('x pixels')
plt.ylabel('y pixels')
plt.title(title_string)
plt.gcf().set(dpi=200)

plt.savefig(scanPath.joinpath(f'raw_map_{blend_name}_{detector}_{energy}eV_pol{int(pol)}.png'), dpi=200)

plt.show()

In [ ]:
# If you need to draw new mask:
draw = PyHyperScattering.IntegrationUtils.DrawMask(mask_img)
draw.ui()

In [ ]:
# Save and load drawn mask
draw.save(maskPath.joinpath(f'scan{scan_id}.json'))
mask = draw.mask

In [ ]:
# Load a previously drawn mask
draw = PyHyperScattering.IntegrationUtils.DrawMask(mask_img)

### WAXS: 
draw.load(maskPath.joinpath('scan44626.json'))

### SAXS:

# ### Other:
# draw.load(maskPath.joinpath('scan43213.json'))

# ### Unique:
# draw.load(maskPath.joinpath(f'scan{scan_id}.json'))


draw.save(maskPath.joinpath(f'scan{scan_id}.json'))
mask = draw.mask

In [ ]:
### Saves masked raw map image

img = mask_img.data.astype('float')
img[mask] = np.nan

cm = plt.cm.terrain.copy()
cm.set_bad('purple')

plt.imshow(img, origin='lower', aspect=1, norm=LogNorm(2e1,5e3), cmap=cm)
# plt.xlim(bcx_approx-300, bcx_approx+300)
# plt.ylim(bcy_approx-300, bcy_approx+300)
ax = plt.gca()
ax.xaxis.set_major_formatter(plt.NullFormatter())
ax.yaxis.set_major_formatter(plt.NullFormatter())
ax.xaxis.set_major_locator(plt.NullLocator())
ax.yaxis.set_major_locator(plt.NullLocator())
plt.colorbar()
plt.title(f'{blend_name}, {detector}, {energy} eV, pol = {int(pol)}$^\circ$')

plt.savefig(scanPath.joinpath(f'raw_map__masked_{blend_name}_{detector}_{energy}eV_pol{int(pol)}.png'), dpi=200)

plt.gcf().set(dpi=200)
plt.show()

In [ ]:
### Saves masked data array as pkl

masked = raw.unstack('system')

data = masked.data.astype('float')
data[mask, :] = np.nan
masked.data = data

pickle.dump(masked, arrPath.joinpath(f'masked_{scan_id}_{blend_name:-<13}_{detector}_{int(pol)}deg.pkl').open('wb'), protocol=-1)

plt.imshow(masked.sel(energy=285, method='nearest'), origin='lower', cmap=cm, norm=LogNorm(2e1, 5e3))
plt.gcf().set(dpi=200)
plt.show()

## Finito for raw

In [ ]:
# Load mask into integrator and check that it is there (along with beam center location)
integ = PyHyperScattering.integrate.PFEnergySeriesIntegrator(geomethod='template_xr', template_xr = mask_img)
integ.mask = mask

PyHyperScattering.IntegrationUtils.Check.checkAll(integ, mask_img, img_max=1e3, alpha=1)
plt.xlim(bcx_approx-200, bcx_approx+200)
plt.ylim(bcy_approx-200, bcy_approx+200)
plt.gcf().set(dpi=120)
plt.show()

### Integrate raw Image!

In [ ]:
integrated = integ.integrateImageStack(raw)

In [ ]:
integrated = integrated.unstack('system')
pickle.dump(integrated, arrPath.joinpath(f'integrated_{scan_id}_{blend_name:-<13}_{detector}_{int(pol)}deg.pkl').open('wb'), protocol=-1)

integrated

In [ ]:
# Option to load from pkl file as well:
scan_id = 43181
integrated = pickle.load(sorted(savePath.glob(f'{scan_id}*.pkl'))[0].open('rb'))

sample_name = integrated.attrs['sample_name']
blend_name = sample_guide[sample_name]
pol = integrated.attrs['polarization'][0]
detector = detector_guide[integrated.attrs['detector']]
print(f'{sample_name}, {blend_name}, {int(pol)}, {detector}')

In [ ]:
integrated.sel(energy=285, method='nearest').plot(norm=LogNorm(1.5e0,9e3), cmap=cmap)
plt.xscale('log')
plt.xlim(waxs_xlim)
plt.xlabel('Q [$nm^{-1}$]')
plt.ylabel('Chi [degrees]')
plt.gcf().set(dpi=150)
plt.title(f'{blend_name}, {detector}, {energy} eV, pol. = {int(pol)}$^\circ$')

plt.savefig(scanPath.joinpath(f'CvsQ_map_{blend_name}_{detector}_{energy}eV_pol{int(pol)}.png'), dpi=200)

plt.show()

In [ ]:
int(pol)

In [ ]:
# test = mpl.ticker.StrMethodFormatter('{x:.0e}')
chi_slice = slice(-45, 45)

integrated.sel(chi=chi_slice).mean('chi').plot(norm=LogNorm(1e0,2e4), cmap=cmap, x='q', 
        xlim=waxs_xlim, ylim=(280,290))

plt.xscale('log')
plt.xlabel('Q [$nm^{-1}$]')
plt.ylabel('Energy [eV]')
plt.gca().yaxis.set_major_locator(plt.MultipleLocator(1))
# plt.gca().xaxis.set_minor_formatter(test)
plt.gcf().set(dpi=100, size_inches=(8,5))

if int(pol) == 0:
    plt.title(f'{blend_name}, {detector}, para: [chi=({chi_slice.start}, {chi_slice.stop}) , pol. = {int(pol)}$^\circ$]')
    plt.savefig(scanPath.joinpath(f'EvsQ_map_{blend_name}_{detector}_para_pol{int(pol)}.png'), dpi=200)
elif int(pol) == 90:
    plt.title(f'{blend_name}, {detector}, perp: [chi=({chi_slice.start}, {chi_slice.stop}) , pol. = {int(pol)}$^\circ$]')
    plt.savefig(scanPath.joinpath(f'EvsQ_map_{blend_name}_{detector}_perp_pol{int(pol)}.png'), dpi=200)   

plt.show()

In [ ]:
# test = mpl.ticker.StrMethodFormatter('{x:.0e}')
chi_slice = slice(-135, -45)

integrated.sel(chi=chi_slice).mean('chi').plot(norm=LogNorm(1e0,2e4), cmap=cmap, x='q', 
        xlim=waxs_xlim, ylim=(280,290))

plt.xscale('log')
plt.xlabel('Q [$nm^{-1}$]')
plt.ylabel('Energy [eV]')
plt.gca().yaxis.set_major_locator(plt.MultipleLocator(1))
# plt.gca().xaxis.set_minor_formatter(test)
plt.gcf().set(dpi=100, size_inches=(8,5))

if int(pol)==0:
    plt.title(f'{blend_name}, {detector}, perp: [chi=({chi_slice.start}, {chi_slice.stop}) , pol. = {int(pol)}$^\circ$]')
    plt.savefig(scanPath.joinpath(f'EvsQ_map_{blend_name}_{detector}_perp_pol{int(pol)}.png'), dpi=200)
elif int(pol)==90:
    plt.title(f'{blend_name}, {detector}, para: [chi=({chi_slice.start}, {chi_slice.stop}) , pol. = {int(pol)}$^\circ$]')
    plt.savefig(scanPath.joinpath(f'EvsQ_map_{blend_name}_{detector}_para_pol{int(pol)}.png'), dpi=200)

plt.show()

In [ ]:
# test = mpl.ticker.StrMethodFormatter('{x:.0e}')

integrated.mean('chi').plot(norm=LogNorm(1e0,2e4), cmap=cmap, x='q', 
        xlim=waxs_xlim, ylim=(280,290))

plt.xscale('log')
plt.xlabel('Q [$nm^{-1}$]')
plt.ylabel('Energy [eV]')
plt.gca().yaxis.set_major_locator(plt.MultipleLocator(1))
# plt.gca().xaxis.set_minor_formatter(test)
plt.gcf().set(dpi=100, size_inches=(8,5))
plt.title(f'{blend_name}, {detector}, full chi average, pol. = {int(pol)}$^\circ$]')

plt.savefig(scanPath.joinpath(f'EvsQ_map_{blend_name}_{detector}_full360_pol{int(pol)}.png'), dpi=200)
plt.show()

In [ ]:
energies = np.arange(282.5, 290, 0.5)
lineplot_cmap = plt.cm.viridis(np.linspace(0,0.9, len(energies)))
energies.shape

In [ ]:
from matplotlib.ticker import LogFormatterSciNotation

In [ ]:
saxs_xlim = (8.5e-4, 1e-2) #(8.5e-4, 1e-2)
waxs_xlim = (9e-3, 1.1e-1)

saxs_ylim = (1e0, 1e4)
waxs_ylim = (1e0, 1e4)

In [ ]:
# Full 360 chi mean
for i, energy in enumerate(energies):
    integrated.mean('chi').sel(energy=energy, method='nearest').plot(label=energy, color=lineplot_cmap[i])

plt.title(f'{blend_name}, {detector}, pol={int(pol)}')    
plt.xscale('log')
plt.xlabel('Q [1/nm]')
plt.xlim(waxs_xlim)
plt.yscale('log')
plt.ylabel('Intensity [arb. units]')
plt.ylim(waxs_ylim)
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.gcf().set_size_inches(8,5)

# plt.gca().xaxis.set_minor_locator(plt.LogLocator(subs='all'))
plt.gca().xaxis.set_minor_formatter(LogFormatterSciNotation(minor_thresholds=(2, 1)))

plt.savefig(scanPath.joinpath(f'IvsQ_plot_{blend_name}_{detector}_full360_pol{int(pol)}.svg'), pad_inches=0.2, bbox_inches='tight')



In [ ]:
# Perp chi mean
for i, energy in enumerate(energies):
    integrated.sel(chi=slice(-135,45)).mean('chi').sel(energy=energy, method='nearest').plot(label=energy, color=lineplot_cmap[i])

plt.xscale('log')
plt.xlabel('Q [1/nm]')
plt.xlim(waxs_xlim)
plt.yscale('log')
plt.ylabel('Intensity [arb. units]')
plt.ylim(waxs_ylim)
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.gcf().set_size_inches(8,5)

# plt.gca().xaxis.set_minor_locator(plt.LogLocator(subs='all'))
plt.gca().xaxis.set_minor_formatter(LogFormatterSciNotation(minor_thresholds=(2, 1)))

if int(pol)==0:
    plt.title(f'{blend_name}, {detector}, perp slice, pol={int(pol)}')    
    plt.savefig(scanPath.joinpath(f'IvsQ_plot_{blend_name}_{detector}_perp_pol{int(pol)}.svg'), pad_inches=0.2, bbox_inches='tight')
elif int(pol)==90:
    plt.title(f'{blend_name}, {detector}, para slice, pol={int(pol)}')    
    plt.savefig(scanPath.joinpath(f'IvsQ_plot_{blend_name}_{detector}_para_pol{int(pol)}.svg'), pad_inches=0.2, bbox_inches='tight')


In [ ]:
# Para chi mean
for i, energy in enumerate(energies):
    integrated.sel(chi=slice(-45,45)).mean('chi').sel(energy=energy, method='nearest').plot(label=energy, color=lineplot_cmap[i])

plt.xscale('log')
plt.xlabel('Q [1/nm]')
plt.xlim(waxs_xlim)
plt.yscale('log')
plt.ylabel('Intensity [arb. units]')
plt.ylim(waxs_ylim)
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.gcf().set_size_inches(8,5)

# plt.gca().xaxis.set_minor_locator(plt.LogLocator(subs='all'))
plt.gca().xaxis.set_minor_formatter(LogFormatterSciNotation(minor_thresholds=(2, 1)))

if int(pol)==0:
    plt.title(f'{blend_name}, {detector}, para slice, pol={int(pol)}')    
    plt.savefig(scanPath.joinpath(f'IvsQ_plot_{blend_name}_{detector}_para_pol{int(pol)}.svg'), pad_inches=0.2, bbox_inches='tight')
elif int(pol)==90:
    plt.title(f'{blend_name}, {detector}, perp slice, pol={int(pol)}')    
    plt.savefig(scanPath.joinpath(f'IvsQ_plot_{blend_name}_{detector}_perp_pol{int(pol)}.svg'), pad_inches=0.2, bbox_inches='tight')  


In [ ]:
integrated.rsoxs.AR(chi_width=90).plot(x='q', vmin=-0.8, vmax=0.8, cmap=plt.cm.seismic)
plt.xlim(waxs_xlim)
plt.ylim(275, 300)
plt.title(f'{blend_name}, {detector}, pol={int(pol)}')    
plt.xscale('log')
plt.gcf().set(dpi=150)
plt.gca().xaxis.set_minor_formatter(plt.LogFormatter(minor_thresholds=(2, 1)))

plt.savefig(scanPath.joinpath(f'AR_map_{blend_name}_{detector}_pol{int(pol)}.png'), dpi=200)

plt.show()

In [ ]:
raw

In [ ]:
np.save('testing.npy', integrated.data)

In [ ]:
integrated.dims

In [ ]:
np.load('testing.npy')

## NEXAFS stuff below:

In [ ]:
df

In [ ]:
df.drop_vars(['time_bins','time']).hvplot.line()

In [ ]:
import seaborn as sns
sns.set()
fig, axes  = plt.subplots(3,1,sharex=True)
for channel,axis in zip(['RSoXS Sample Current','SAXS Beamstop','WAXS Beamstop'],axes):
    axis.plot(df['en'],df[channel],label=channel)
    axis.fill_between(df['en'],df[channel]+df[channel+'_std'],df[channel]-df[channel + '_std'],alpha=0.5)
    axis.legend(title='')
plt.show()

In [ ]:
plt.close()

In [ ]:
i0df['i0correction'].interp(en=[300,301])

In [ ]:
timeleftedges[1,40]-timeleftedges[0]

In [ ]:
df.hvplot.scatter(x='en',y=['RSoXS Au Mesh Current','WAXS Beamstop','SAXS Beamstop','RSoXS Sample Current'])

In [ ]:
df.hvplot.errorbars(x='en',y='SAXS Beamstop', yerr1='SAXS Beamstop_std')

In [ ]:
list(newdf.data_vars.keys())[0]

In [ ]:
c[39226].primary.data['en_monoen_readback']

In [ ]:
plot_processed_NEXAFS(39226)

In [ ]:
grouped_df.hvplot(x='energy')

In [ ]:
iodf.hvplot.scatter(y='i0correction')

In [ ]:
groupeddata = df.groupby_bins('energy',run.primary.data['en_energy']-0.05,squeeze=True).mean()
groupeddata = df.groupby_bins('energy',run.primary.data['en_energy']-0.05,squeeze=True).mean()

groupeddata = groupeddata.assign_coords(energy = ('energy_bins',[value.mid for value in groupeddata.energy_bins.values]))
groupeddata = groupeddata.swap_dims({'energy_bins':'energy'})

In [ ]:
groupeddata.hvplot.scatter()

In [ ]:
centers = [value.mid for value in groupeddata.energy_bins.values]

In [ ]:
centers

In [ ]:
lastscan.start['RSoXS_Main_DET']

In [ ]:
%%time
make_monitor_html(39152)

In [ ]:
stream_summary(39152)
#find_scans(plan='spiralsearch')

In [ ]:
%%time
make_raw_2D_html(39226)

In [ ]:
#%time coarsened.plot.imshow(x='pix_x',y='pix_y',col='energy',=10,norm=LogNorm(1,6e6),cmap=cmap)

In [ ]:
def live_reduce(RUN_TO_PLOT,max_intensity = 1e5,return_data = False,prior_runs=None,loader = None,catalog = None,integrator=None):
    '''
    Example live-analysis function
    
    Parameters:
        run_to_plot (int): the local scan id from DataBroker
    '''

    if loader is None:
        rsoxsload = PyHyperScattering.load.SST1RSoXSDB(corr_mode='none')
    else:
        rsoxsload = loader
    if catalog is None:
        c = from_profile('rsoxs')
    else:
        c = catalog


    itp = rsoxsload.loadRun(c[RUN_TO_PLOT],dims=['energy'])

    name = itp.attrs["sample_name"]
    scan_id = itp.attrs['start']['scan_id']
    print(f'Processing {scan_id} / {name} started @ {itp.attrs["start"]["time"]}, {prior_runs}')

    if prior_runs is not None:
        print('doing prior run check')
        try:
            prev_len = prior_runs[scan_id]
            if len(itp) == prev_len:
                print(f'Already have {prev_len} images for {scan_id}, not going any further.')
                return prior_runs
            else:
                prior_runs[scan_id] = len(itp)
        except KeyError:
            print('keyerror on check, inserting')
            prior_runs[scan_id] = len(itp)
    print(prior_runs)
            
    if 'energy' in itp.indexes['system'].names:
        itp.unstack('system').plot.imshow(x='pix_x',y='pix_y',col='energy',col_wrap=5,norm=LogNorm(1,max_intensity))
    elif 'sam_x' in itp.indexes['system'].names:
        itp.unstack('system').plot.imshow(x='pix_x',y='pix_y',col='sam_y',row='sam_x',norm=LogNorm(1,max_intensity))
    else:
        itp.plot.imshow(x='pix_x',y='pix_y',col='system',norm=LogNorm(1,max_intensity))

    plt.title(f'{scan_id} / {name} raw')
    plt.savefig(f'rawcomp_{scan_id}_{name}.png')
    plt.savefig(f'rawcomp_latest.png')
    plt.clf()
    if not os.path.exists(f'AR_{scan_id}_{name}.png'):
        print('attempting reduction')
        try:
            if itp.attrs['stop']['exit_status'] == 'success':
                if itp.rsoxs_config == 'waxs':
                    maskmethod = 'nika'
                    mask = '/nsls2/data/sst1/legacy/RSoXS/analysis/SST1_WAXS_mask.hdf'
                elif itp.rsoxs_config == 'saxs':
                    maskmethod = 'nika'
                    mask = '/nsls2/data/sst1/legacy/RSoXS/analysis/SST1-SAXS_mask.hdf'
                else:
                    maskmethod = 'none'
                    warnings.warn(f'Bad rsoxs_config, expected saxs or waxs but found {itp.rsoxs_config}.  This will disable masking and certainly cause issues later.',stacklevel=2)

                if integrator is None:
                    integ = PyHyperScattering.integrate.PFEnergySeriesIntegrator(maskmethod=maskmethod,maskpath=mask,geomethod='template_xr',template_xr=itp,integration_method='csr_ocl')
                else:
                    integ = integrator

                integratedimages = integ.integrateImageStack(itp)
                try:
                    integratedimages.fileio.saveNexus(f'reduced_{scan_id}_{name}.nxs')
                except:
                    integratedimages.fileio.savePickle(f'reduced_{scan_id}_{name}.p')

                integratedimages = integratedimages.unstack('system')
                try:
                    integratedimages.sel(energy=270).plot(norm=LogNorm(1,max_intensity))
                    plt.title(f'{name} @ 270 eV')
                    plt.savefig(f'270_{scan_id}_{name}.png')
                    plt.savefig(f'270_latest.png')
                    plt.clf()
                except:
                    pass
                
                #if 'energy' in integratedimages.indexes:
                print(len(integratedimages.energy))
                if len(integratedimages.energy) > 2:
                    integratedimages.rsoxs.AR().plot(vmin=-1,vmax=1,cmap='RdBu')
                    plt.title(f'{name} AR')
                    plt.savefig(f'AR_{scan_id}_{name}.png')
                    plt.savefig(f'AR_latest.png')
                    plt.clf()

                    integratedimages.mean('chi').plot(norm=LogNorm(1,max_intensity))
                    plt.title(f'{name} IqE')
                    plt.savefig(f'IqE_{scan_id}_{name}.png')
                    plt.savefig(f'IqE_latest.png')
                    plt.clf()

                if return_data:
                    return integratedimages

        except KeyError:
            warnings.warn('Attempted to reduce an incomplete scan.',stacklevel=2)
    print(prior_runs)
    if prior_runs is not None:
        print('reached return')
        return prior_runs


In [ ]:
live_reduce(-1)

In [ ]:
import time
prior_runs = {}
while True:
    try:
        make_monitor_html(-1)
        make_raw_2D_html(-1)
    except Exception as e:
        print(e)
        pass
    try:
        prior_runs = live_reduce(-1,prior_runs=prior_runs)
    except Exception as e:
        print(e)
        print(e.args)
        
        if 'Try again' in str(e):
            pass
        else:
            raise e

    print(f'Prior runs: {prior_runs}')
    time.sleep(5)